# Contraceptive Method Choice: Data Generation (Multivariate)

this notebook loads the Contraceptive Method Choice dataset from the UCI repository, and generates three datasets with MCAR, MAR, and MNAR missingness using mdatagen.

run this notebook once to reproduce the CSV files used by `../analysis_multi.ipynb`. the CSVs are saved in the same `data/` folder as this notebook.

In [1]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

from mdatagen.multivariate.mMCAR import mMCAR
from mdatagen.multivariate.mMAR  import mMAR
from mdatagen.multivariate.mMNAR import mMNAR

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
# initial data set loading
cmc = fetch_ucirepo(id=30)
X_raw = cmc.data.features
y = cmc.data.targets.iloc[:, 0].to_numpy()

X_raw.isna().sum()

X_raw.shape

wife_age                    0
wife_edu                    0
husband_edu                 0
num_children                0
wife_religion               0
wife_working                0
husband_occupation          0
standard_of_living_index    0
media_exposure              0
dtype: int64

(1473, 9)

The dataset is already complete with no missing values.

We encode any categorical columns as integer codes so that every column is numeric, required by the mdatagen generators.

In [3]:
X = X_raw.copy()
cat_cols = X.select_dtypes(include=["object", "category"]).columns
for col in cat_cols:
    X[col] = pd.factorize(X[col])[0]
X = X.astype(float)

X.isna().sum()

X.shape

wife_age                    0
wife_edu                    0
husband_edu                 0
num_children                0
wife_religion               0
wife_working                0
husband_occupation          0
standard_of_living_index    0
media_exposure              0
dtype: int64

(1473, 9)

Now we have a clean numeric dataset to work with. We inject missing values into multiple columns simultaneously to simulate realistic multivariate missingness patterns.

We use `contraceptive_method_used` as the target variable `y` (1=no use, 2=long-term, 3=short-term). It is passed to the generators because the library requires it, `contraceptive_method_used` is not a feature in `X` and will not receive any missing values.

### MCAR generation

Here we'll generate a dataset with MCAR missingness across multiple columns simultaneously.

Missingness is completely random and does not depend on any observed variable or on `contraceptive_method_used`.

In [4]:
np.random.seed(42)
df_mcar = mMCAR(X=X, y=y, missing_rate=30).random()

df_mcar.isna().sum()

df_mcar.shape

wife_age                    437
wife_edu                    442
husband_edu                 441
num_children                456
wife_religion               435
wife_working                447
husband_occupation          430
standard_of_living_index    447
media_exposure              442
dtype: int64

(1473, 9)

### MAR generation

Here we'll generate a dataset with MAR missingness across multiple columns simultaneously.

Missingness is introduced using the random method from mdatagen. In `n_xmiss=4` randomly selected column pairs, the rows with the lowest values of the observed column are made missing in the other column, so missingness depends on observed values, not on the missing value itself.

In [5]:
np.random.seed(42)
df_mar = mMAR(X=X, y=y, n_xmiss=4).random(missing_rate=30)

df_mar.isna().sum()

df_mar.shape

wife_age                    994
wife_edu                      0
husband_edu                   0
num_children                  0
wife_religion               994
wife_working                994
husband_occupation          994
standard_of_living_index      0
media_exposure                0
target                        0
dtype: int64

(1473, 10)

### MNAR generation

Here we'll generate a dataset with MNAR missingness across multiple columns simultaneously.

Missingness is generated on the prediction target `contraceptive_method_used` using `threshold=1`. Rows at the lower end of the target distribution are most likely to have their feature records missing, simulating the case where certain groups are systematically underrepresented in the data.

Since `contraceptive_method_used` is the target (`y`) and is absent from the generated feature matrix, its effect must be inferred through correlated proxies in the feature columns.

In [6]:
np.random.seed(42)
df_mnar = mMNAR(X=X, y=y, threshold=1, n_xmiss=4).random(missing_rate=30)

df_mnar.isna().sum()

df_mnar.shape

wife_age                    994
wife_edu                      0
husband_edu                   0
num_children                  0
wife_religion                 0
wife_working                994
husband_occupation            0
standard_of_living_index    994
media_exposure              994
target                        0
dtype: int64

(1473, 10)

Now we save the generated datasets to CSV so the analysis notebook can load them any time.

In [7]:
df_mcar.to_csv("data/mcar_multi.csv", index=False)
df_mar.to_csv("data/mar_multi.csv", index=False)
df_mnar.to_csv("data/mnar_multi.csv", index=False)